In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# GoEmotions  (test 2)
# ------------
# Emotion recognition of twitter dataset using
# HuggingFace Transformers ("go_emotions" dataset)
#
# contains a train/test/validation split:
# Size of training dataset: 43,410.
# Size of test dataset: 5,427.
# Size of validation dataset: 5,426.
#
# The 27 emotion categories are:
# admiration, amusement, anger, annoyance, approval, caring, confusion, curiosity, desire,
# disappointment, disapproval, disgust, embarrassment, excitement, fear, gratitude, grief,
# joy, love, nervousness, optimism, pride, realization, relief, remorse, sadness, surprise.
#

In [12]:
!pip install -U transformers
!pip install -U datasets
!pip install -U accelerate

zsh:1: command not found: pip
zsh:1: command not found: pip
zsh:1: command not found: pip


In [1]:
import os
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm

In [22]:
#model_path = "./go_emotions_trained_model"

In [2]:
from datasets import load_dataset
goemotions = load_dataset("go_emotions")
print(goemotions)

/Users/jessie_guo/miniconda3/envs/Python3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})


In [3]:
print(goemotions['train'].features)
classes = goemotions['train'].features['labels'].feature.names
print(f'\nClasses: {classes}')

{'text': Value('string'), 'labels': List(ClassLabel(names=['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral'])), 'id': Value('string')}

Classes: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [4]:
#Tokenization and Preprocessing
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [9]:
def tokenize_and_process_labels(examples):
    # Tokenize the text
    tokenized_inputs = tokenizer(examples["text"], 
                                 padding="max_length", 
                                 truncation=True)

    # Process labels for multi-label classification (one-hot encoding)
    labels_batch = []
    for labels_list in examples["labels"]:
        one_hot_labels = np.zeros(len(classes), dtype=float)
        for label_idx in labels_list:
            one_hot_labels[label_idx] = 1.0
        labels_batch.append(one_hot_labels.tolist()) # Convert to list for dataset

    tokenized_inputs["labels"] = labels_batch
    return tokenized_inputs

tokenized_datasets = goemotions.map(tokenize_and_process_labels, batched=True)
print(pd.DataFrame(tokenized_datasets['train']))

# Explicitly set the format for the labels to be torch.float
# This ensures Trainer treats it as multi-label classification
tokenized_datasets.set_format("torch", columns=['input_ids', 'attention_mask', 'labels'], output_all_columns=True)
print(pd.DataFrame(tokenized_datasets['train']))

                                                    text  \
0      My favourite food is anything I didn't have to...   
1      Now if he does off himself, everyone will thin...   
2                         WHY THE FUCK IS BAYLESS ISOING   
3                            To make her feel threatened   
4                                 Dirty Southern Wankers   
...                                                  ...   
43405  Added you mate well I’ve just got the bow and ...   
43406  Always thought that was funny but is it a refe...   
43407  What are you talking about? Anything bad that ...   
43408            More like a baptism, with sexy results!   
43409                                    Enjoy the ride!   

                                                  labels       id  \
0      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  eebbqej   
1      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  ed00q6i   
2      [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  eezlygj   
3  

In [13]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels = len(classes))
model.config.problem_type = "multi_label_classification"

Loading weights: 100%|████████████████████| 100/100 [00:00<00:00, 11980.64it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
from transformers import TrainingArguments

batch_size = 32 # Reduced batch size to mitigate OutOfMemoryError
model_name = 'distilbert_finetuned-goemotions'

training_args = TrainingArguments(output_dir = model_name,
                                  num_train_epochs=2,
                                  learning_rate=2e-5,
                                  per_device_train_batch_size=batch_size,
                                  per_device_eval_batch_size=batch_size,
                                  weight_decay=0.01,
                                  eval_strategy='epoch',
                                  disable_tqdm=False)

In [15]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
import torch.nn as nn

# Verify model problem type right before Trainer instantiation
print(f"Model problem type before Trainer: {model.config.problem_type}")

class CustomDataCollator(DataCollatorWithPadding):
    def __call__(self, features):
        batch = super().__call__(features) #super () = call parent class's __call__, 
        #__call__ allows object to behave like a function, line lets extend existing functionality
        # Ensure labels are float tensors for multi-label classification
        if "labels" in batch:
            batch["labels"] = batch["labels"].to(torch.float)
        return batch

data_collator = CustomDataCollator(tokenizer=tokenizer)

trainer = Trainer(model=model,
                  args=training_args,
                  train_dataset=tokenized_datasets["train"],
                  eval_dataset=tokenized_datasets["validation"],
                  data_collator=data_collator)
trainer.train()

Model problem type before Trainer: multi_label_classification


/Users/jessie_guo/miniconda3/envs/Python3.10/lib/python3.10/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG16GFamilyCommandBuffer: 0x3587cab10>
    label = <none> 
    device = <AGXG16GDevice: 0x16b997800>
        name = Apple M4 
    commandQueue = <AGXG16GFamilyCommandQueue: 0x1699cea00>
        label = <none> 
        device = <AGXG16GDevice: 0x16b997800>
            name = Apple M4 
    retainedReferences = 1
Error: command buffer exited with error status.
	The Metal Performance Shaders operations encoded on it may not have completed.
	Error: 
	(null)
	Insufficient Memory (00000008:kIOGPUCommandBufferCallbackErrorOutOfMemory)
	<AGXG16GFamilyCommandBuffer: 0x357dfa140>
    label = <none> 
    device = <AGXG16GDevice: 0x16b997800>
        name = Apple M4 
    commandQueue = <AGXG16GFamilyCommandQueue: 0x1699cea00>
        label = <none> 
        device = <AGXG16GDev

KeyboardInterrupt: 

In [30]:
#Zip the folder 'distilbert_finetuned-goemotions' in Colab
!zip -r /content/drive/MyDrive/Colab_Notebooks/distilbert_finetuned-goemotions.zip /content/drive/MyDrive/Colab_Notebooks/distilbert_finetuned-goemotions

  adding: content/drive/MyDrive/Colab_Notebooks/distilbert_finetuned-goemotions/ (stored 0%)
  adding: content/drive/MyDrive/Colab_Notebooks/distilbert_finetuned-goemotions/checkpoint-2500/ (stored 0%)
  adding: content/drive/MyDrive/Colab_Notebooks/distilbert_finetuned-goemotions/checkpoint-2500/rng_state.pth (deflated 26%)
  adding: content/drive/MyDrive/Colab_Notebooks/distilbert_finetuned-goemotions/checkpoint-2500/optimizer.pt (deflated 23%)
  adding: content/drive/MyDrive/Colab_Notebooks/distilbert_finetuned-goemotions/checkpoint-2500/scheduler.pt (deflated 61%)
  adding: content/drive/MyDrive/Colab_Notebooks/distilbert_finetuned-goemotions/checkpoint-2500/tokenizer.json (deflated 71%)
  adding: content/drive/MyDrive/Colab_Notebooks/distilbert_finetuned-goemotions/checkpoint-2500/tokenizer_config.json (deflated 42%)
  adding: content/drive/MyDrive/Colab_Notebooks/distilbert_finetuned-goemotions/checkpoint-2500/trainer_state.json (deflated 62%)
  adding: content/drive/MyDrive/Cola

In [31]:
#Download the zip file distilbert_finetuned-goemotions.zip from Colab
from google.colab import files
files.download("/content/drive/MyDrive/Colab_Notebooks/distilbert_finetuned-goemotions.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# load the pretrained model later:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

path = "distilbert_finetuned-goemotions/checkpoint-500"
model_new = AutoModelForSequenceClassification.from_pretrained(path)
tokenizer_new = AutoTokenizer.from_pretrained(path)

In [ ]:
#text = 'I love Machine Learning! Tokenization is fun.'
#text = 'I love you'
text = 'I hate you'

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

input_encoded = tokenizer_new(text, return_tensors='pt').to(device)
with torch.no_grad():
    outputs = model_new(**input_encoded)

logits = outputs.logits # logits are predictions
pred = torch.argmax(logits, dim=1).item()
print(pred, classes[pred])
